Importing and Intializing everything

In [ ]:
import pandas as pd
import numpy as np
import holidays
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, confusion_matrix, accuracy_score

# Policy: Takes in the last WINDOW days of AQI + pollutant data and predicts tomorrow's AQI.
# Uses daily data only, pooled across all cities. Date is treated purely as a date.
# Short gaps (<= MAX_GAP days) in the INPUT features are forward-filled; the target is never filled.
# Classification: predicted AQI is binned into the Indian (CPCB) AQI categories.
# Feature reduction (correlation filter / PCA) is fit on the TRAINING data only, then applied to test.

WINDOW = 7          # number of past days (including today) fed to the model
MAX_GAP = 2         # forward-fill at most this many consecutive missing days in features
RIDGE_ALPHA = 10.0
TEST_FRACTION = 0.2
pollutants = ['PM2.5', 'PM10', 'NO', 'NO2', 'NOx', 'NH3', 'CO', 'SO2', 'O3', 'Benzene', 'Toluene', 'Xylene']
# Edit the list above to drop sparse pollutants (an empty list gives an AQI-only model)
base_cols = ['AQI'] + pollutants

# Feature reduction settings
REDUCTION_STRATEGIES = ['none', 'corr', 'pca', 'corr+pca']  # remove any you don't want to run
CORR_THRESHOLD = 0.95    # drop a lag feature if |correlation| with an earlier kept feature exceeds this
PCA_VARIANCE = 0.95      # PCA keeps enough components to explain this fraction of variance
SHOW_CONFUSION_MATRICES = True

# Indian (CPCB) AQI categories
BINS = [-np.inf, 50, 100, 200, 300, 400, np.inf]
ORDER = ['Good', 'Satisfactory', 'Moderate', 'Poor', 'Very Poor', 'Severe']

def to_bucket(aqi_values):
    """Convert continuous AQI values into CPCB category labels."""
    return pd.cut(np.asarray(aqi_values, dtype=float), bins=BINS, labels=ORDER).astype(str)

Data cleaning and filtering

In [ ]:
# ---------- Load & clean ----------
df = pd.read_csv('AQIData2015to20/city_day.csv')
print(len(df), 'rows,', df['AQI'].isnull().sum(), 'with empty AQI')

df['Date'] = pd.to_datetime(df['Date'], format = '%d-%m-%Y')
df = df.drop_duplicates(subset=['City', 'Date']).sort_values(['City', 'Date'])

# ---------- Discard invalid AQI values ----------
AQI_MAX = 500
invalid = (df['AQI'] > AQI_MAX) | (df['AQI'] < 0)
print(f'Discarding {invalid.sum()} rows with AQI > {AQI_MAX} or AQI < 0 '
      f'({(df["AQI"] > AQI_MAX).sum()} above, {(df["AQI"] < 0).sum()} below)')
df = df[~invalid]

# Reindex each city to a complete daily calendar so missing days become NaN rows.
# This guarantees "lag 3" really means 3 days ago, not "3 rows ago".
frames = []
for city, g in df.groupby('City'):
    g = g.set_index('Date')[base_cols].asfreq('D')
    g['City'] = city
    frames.append(g)
df = pd.concat(frames).reset_index().sort_values(['City', 'Date']).reset_index(drop=True)

# ---------- Target (from RAW AQI, never filled) ----------
g = df.groupby('City')
df['AQI_next'] = g['AQI'].shift(-1)

# ---------- Fill short gaps in input features (past values only, so no leakage) ----------
filled = g[base_cols].ffill(limit=MAX_GAP)
filled['City'] = df['City']
gf = filled.groupby('City')

Builidng Features and functions for feature reduction

In [ ]:

# ---------- Features: last WINDOW days of AQI + pollutants ----------
# lag0 = today, lag1 = yesterday, ...
feats = {}
for lag in range(WINDOW):
    for col in base_cols:
        feats[f'{col}_lag{lag}'] = gf[col].shift(lag)
feats['AQI_roll'] = gf['AQI'].transform(lambda s: s.rolling(WINDOW, min_periods=1).mean())

data = pd.concat([df[['City', 'Date', 'AQI_next']], pd.DataFrame(feats)], axis=1)

print('Rows with tomorrow\'s AQI known:', data['AQI_next'].notna().sum())
data = data.dropna()
print('Usable samples after building windows:', len(data))
print('Usable samples per city:')
print(data['City'].value_counts().to_string())
lost = set(df['City'].unique()) - set(data['City'].unique())
if lost:
    print('Cities with zero usable samples (check their pollutant coverage):', sorted(lost))

# ---------- Calendar features for the day being predicted (tomorrow) ----------
next_day = data['Date'] + pd.Timedelta(days=1)
data['Weekday'] = next_day.dt.dayofweek
data['is_weekend'] = next_day.dt.dayofweek.isin([5, 6]).astype(int)
data['month_sin'] = np.sin(2 * np.pi * next_day.dt.month / 12)
data['month_cos'] = np.cos(2 * np.pi * next_day.dt.month / 12)
india_holidays = holidays.India(years=next_day.dt.year.unique().tolist())
data['is_festival'] = next_day.dt.date.apply(lambda d: 1 if d in india_holidays else 0)

# ---------- X / y (City one-hot encoded) ----------
city_dummies = pd.get_dummies(data['City'], prefix='City', drop_first=True, dtype=int)
X = pd.concat([data.drop(columns=['City', 'Date', 'AQI_next']), city_dummies], axis=1)
y = data['AQI_next']

# ---------- Chronological split by DATE (not by row, since cities are pooled) ----------
unique_dates = np.sort(data['Date'].unique())
cutoff = unique_dates[int(len(unique_dates) * (1 - TEST_FRACTION))]
train_mask = data['Date'] < cutoff
X_train, X_test = X[train_mask], X[~train_mask]
y_train, y_test = y[train_mask], y[~train_mask]
print(f'Train: {len(X_train)} rows | Test: {len(X_test)} rows | Test starts {pd.Timestamp(cutoff).date()}')

# ---------- Feature reduction helpers ----------
# The correlation filter only considers the lag/rolling features; city dummies and calendar
# features are always kept.
filter_candidates = [c for c in X.columns if '_lag' in c or c == 'AQI_roll']

def correlation_filter(X_tr, candidates, threshold):
    """Return the columns to keep. For every pair of candidate features with
    |corr| > threshold (computed on TRAINING data only), the later column is dropped."""
    corr = X_tr[candidates].corr().abs()
    upper = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1))
    to_drop = [c for c in upper.columns if (upper[c] > threshold).any()]
    return [c for c in X_tr.columns if c not in to_drop], to_drop

Running the model and evaluation

In [ ]:

# ---------- Evaluation helper ----------
y_test_bucket = to_bucket(y_test)

def evaluate(label, y_true, y_pred):
    # Regression metrics
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))

    # Classification: bucket the predicted AQI using Indian standards
    pred_bucket = to_bucket(y_pred)
    acc = accuracy_score(y_test_bucket, pred_bucket)

    print(f"\n{'-' * 60}\n{label}\n{'-' * 60}")
    print("-- Regression --")
    print(f"R square: {r2:.4f}")
    print(f"MAE:      {mae:.2f}")
    print(f"RMSE:     {rmse:.2f}")
    print("-- Classification (AQI bucket) --")
    print(f"Accuracy: {acc:.4f}")
    if SHOW_CONFUSION_MATRICES:
        cm = confusion_matrix(y_test_bucket, pred_bucket, labels=ORDER)
        cm_df = pd.DataFrame(cm,
                             index=pd.Index(ORDER, name='Actual'),
                             columns=pd.MultiIndex.from_product([['Predicted'], ORDER]))
        print("Confusion matrix (rows = actual, columns = predicted):")
        print(cm_df.to_string())

    return {'Model': label, 'R2': r2, 'MAE': mae, 'RMSE': rmse, 'Accuracy': acc}

# ---------- Run every reduction strategy with both models ----------
results = []

for strategy in REDUCTION_STRATEGIES:
    use_corr = 'corr' in strategy
    use_pca = 'pca' in strategy

    # Step 1: correlation filter (fit on training data only)
    if use_corr:
        keep_cols, dropped = correlation_filter(X_train, filter_candidates, CORR_THRESHOLD)
    else:
        keep_cols, dropped = list(X_train.columns), []
    Xtr, Xte = X_train[keep_cols], X_test[keep_cols]

    print(f"\n{'=' * 60}\nFeature reduction: {strategy}\n{'=' * 60}")
    print(f"Features before: {X_train.shape[1]} | after correlation filter: {len(keep_cols)} "
          f"(dropped {len(dropped)})")

    for reg_name, regressor in [('Linear Regression', LinearRegression()),
                                ('Ridge Regression', Ridge(alpha=RIDGE_ALPHA))]:
        # Step 2: scaling, then optional PCA (both fit inside the pipeline on training data only)
        steps = [StandardScaler()]
        if use_pca:
            steps.append(PCA(n_components=PCA_VARIANCE))
        steps.append(regressor)
        model = make_pipeline(*steps)
        model.fit(Xtr, y_train)

        label = f'{reg_name} [{strategy}]'
        if use_pca:
            n_comp = model.named_steps['pca'].n_components_
            print(f"\n{label}: PCA kept {n_comp} components "
                  f"({PCA_VARIANCE:.0%} of variance) from {len(keep_cols)} features")
        results.append(evaluate(label, y_test, model.predict(Xte)))

# Baseline: "tomorrow's AQI = today's AQI"
results.append(evaluate('Baseline (persistence)', y_test, X_test['AQI_lag0']))

# ---------- Summary comparison ----------
print(f"\n{'=' * 60}\nSummary\n{'=' * 60}")
summary = pd.DataFrame(results).set_index('Model')
print(summary.round(4).to_string())